<a href="https://colab.research.google.com/github/yazandarwish264-glitch/m4u3-construction-detection/blob/main/notebooks/02_baseline_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a># 02 — Baseline Inference and Evidence Pack**Construction element detection for site progress verification · MAICEN0526 M4U3 · Group 6**Three things, in order:1. **The baseline.** Run the stock COCO-pretrained YOLOv8 on our site images and show what it does *not* see. This is the argument for the whole project: a general-purpose model is not a construction model.2. **Validation inference.** Our trained model on 10 validation images, side by side with ground truth.3. **New-image inference.** Our trained model on 5 images it has never seen and that were never in any split — the real generalisation test.Everything is written to `results/evidence/`.---### Before you run- **GPU is optional here.** Inference on 15 images runs fine on CPU; it is just slower.- **A Roboflow API key is needed only for part 2** (the validation images live in the dataset). Part 1 and part 3 run without one. If you do not want to enter a key, set `RUN_VALIDATION = False` in cell 1.- `Runtime → Restart session and run all`.**Expected runtime:** 3–6 minutes.

---## 1 · Configuration

In [ ]:
# ---- CONFIG ----------------------------------------------------------------# Our trained weights, from the GitHub Release. Must match notebook 01.WEIGHTS_URL = "PASTE_GITHUB_RELEASE_ASSET_URL_HERE"# This repository, so the notebook can fetch data/new_images/REPO_URL = "https://github.com/yazandarwish264-glitch/m4u3-construction-detection.git"# Roboflow - needed only for validation inference. Already set.RUN_VALIDATION = TrueRF_WORKSPACE   = "yazan-darwish"RF_PROJECT     = "construction-site-km7bh-fapwu"RF_VERSION     = 1# Inference settingsCONF_THRESHOLD = 0.25     # Ultralytics default; see governance_checklist.md section 4IOU_THRESHOLD  = 0.45IMGSZ          = 640N_VALIDATION   = 10       # assignment requires 10N_NEW_IMAGES   = 5        # assignment requires 5SEED           = 0# ---- END CONFIG ------------------------------------------------------------CLASSES = ["brick", "excavator", "pvcpipe", "scaffold", "steelbar"]print("Config loaded.")

---## 2 · Install and set up

In [ ]:
!pip install -q ultralytics roboflowimport os, sys, shutil, random, subprocess, json, urllib.requestfrom pathlib import Pathimport cv2, numpy as npimport matplotlib.pyplot as pltfrom ultralytics import YOLOimport ultralytics, torchrandom.seed(SEED)ROOT     = Path("/content/m4u3")RESULTS  = ROOT / "results"EVIDENCE = RESULTS / "evidence"for d in [EVIDENCE / "baseline", EVIDENCE / "validation", EVIDENCE / "new_images"]:    d.mkdir(parents=True, exist_ok=True)os.chdir(ROOT)print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__}")print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

### 2.1 · Fetch the repositoryGets `data/new_images/` — the unseen images for part 3.**If `data/new_images/` is empty**, put 5 construction photographs in it and commit them before running this. They must be images that were never in the training or validation split. Photos you took yourself are ideal: that is the honest generalisation test, and it is also what the unit asks for.

In [ ]:
repo_dir = ROOT / "repo"NEW_IMAGES_DIR = Noneif "GITHUB_USER" in REPO_URL or not REPO_URL.startswith("https://"):    print("REPO_URL is not set — skipping clone.")    print("Upload your 5 new images manually with the cell below instead.")else:    if repo_dir.exists():        shutil.rmtree(repo_dir)    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)],                       capture_output=True, text=True)    if r.returncode == 0:        NEW_IMAGES_DIR = repo_dir / "data" / "new_images"        n = len([p for p in NEW_IMAGES_DIR.glob("*")                 if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]) if NEW_IMAGES_DIR.exists() else 0        print(f"Cloned. data/new_images/ contains {n} image(s).")        if n == 0:            print("  -> empty. Use the manual-upload cell below.")            NEW_IMAGES_DIR = None    else:        print("Clone failed:", r.stderr[-400:])

#### Manual upload fallbackRun this cell **only** if the clone did not provide the new images. It opens a file picker.

In [ ]:
# Run only if NEW_IMAGES_DIR is None.if NEW_IMAGES_DIR is None:    upload_dir = ROOT / "new_images_uploaded"    upload_dir.mkdir(exist_ok=True)    try:        from google.colab import files        print("Select 5 images that were NOT in the training or validation set.")        uploaded = files.upload()        for fn in uploaded:            shutil.move(fn, upload_dir / fn)        NEW_IMAGES_DIR = upload_dir        print(f"\n{len(list(upload_dir.glob('*')))} image(s) ready.")    except Exception as e:        print(f"Upload unavailable: {e}")else:    print("New images already available — skip this cell.")

---## 3 · Load our trained model

In [ ]:
weights_path = ROOT / "best.pt"if "PASTE" in WEIGHTS_URL:    raise RuntimeError(        "WEIGHTS_URL is not set.\n"        "Create a GitHub Release, attach best.pt from notebook 01, "        "and paste the asset URL into cell 1."    )print(f"Downloading weights from {WEIGHTS_URL} ...")urllib.request.urlretrieve(WEIGHTS_URL, weights_path)import hashlibsha = hashlib.sha256(weights_path.read_bytes()).hexdigest()model = YOLO(str(weights_path))print(f"Loaded.  {weights_path.stat().st_size/1e6:.1f} MB")print(f"SHA-256: {sha}")print(f"Classes: {model.names}")print()print(">>> This SHA must match the one in README section 7. If it does not,")print(">>> the released weights are not the ones you evaluated.")

---## 4 · The baseline — why a general model is not enoughStock `yolov8n.pt`, trained on COCO's 80 everyday classes, run on our site images.COCO contains *toothbrush*, *hair drier* and *teddy bear*. It does not contain reinforcement steel, blockwork, PVC conduit or scaffolding. This cell makes that concrete rather than asserting it — and it is the reason the custom dataset exists.

In [ ]:
baseline = YOLO("yolov8n.pt")print(f"COCO classes available: {len(baseline.names)}")target_terms = ["rebar", "steelbar", "brick", "block", "pvc", "pipe",                "conduit", "scaffold", "excavator", "helmet", "vest"]found = {t: [n for n in baseline.names.values() if t in n.lower()] for t in target_terms}print()print("Does COCO have a class for what we care about?")print("-" * 52)for t, hits in found.items():    print(f"  {t:<12} {'YES: ' + ', '.join(hits) if hits else 'NO'}")print("-" * 52)print("\nNot one of our five classes exists in COCO.")print("A COCO model on a construction site is not inaccurate. It is blind.")

In [ ]:
# Side-by-side on the same images: stock COCO vs our trained model.def annotate(m, img_path, conf):    res = m.predict(source=str(img_path), conf=conf, iou=IOU_THRESHOLD,                    imgsz=IMGSZ, verbose=False)[0]    return cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB), respool = []if NEW_IMAGES_DIR and NEW_IMAGES_DIR.exists():    pool = sorted([p for p in NEW_IMAGES_DIR.glob("*")                   if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])if pool:    sample = pool[:3]    fig, axes = plt.subplots(len(sample), 2, figsize=(14, 5.5 * len(sample)))    axes = np.atleast_2d(axes)    for r, ip in enumerate(sample):        bim, bres = annotate(baseline, ip, CONF_THRESHOLD)        oim, ores = annotate(model, ip, CONF_THRESHOLD)        axes[r, 0].imshow(bim); axes[r, 0].axis("off")        axes[r, 0].set_title(f"Stock COCO YOLOv8n — {len(bres.boxes)} detections", fontsize=10)        axes[r, 1].imshow(oim); axes[r, 1].axis("off")        axes[r, 1].set_title(f"Our model — {len(ores.boxes)} detections", fontsize=10)        cv2.imwrite(str(EVIDENCE / "baseline" / f"baseline_{ip.name}"),                    cv2.cvtColor(bim, cv2.COLOR_RGB2BGR))    plt.tight_layout()    plt.savefig(EVIDENCE / "baseline" / "baseline_comparison.png", dpi=110, bbox_inches="tight")    plt.show()    print("Saved -> results/evidence/baseline/")else:    print("No new images available yet — skipping the visual comparison.")    print("The class-list evidence above still makes the point.")

---## 5 · Validation inference — 10 imagesOur model against the held-out validation split, shown beside ground truth.**This is where your error analysis comes from.** Look at every pair. Find the boxes that are in the prediction but not the truth (false positives) and in the truth but not the prediction (false negatives). Note the filenames. Those six cases go into `docs/error_analysis.md`.

In [ ]:
val_dir = Noneif RUN_VALIDATION:    from getpass import getpass    from roboflow import Roboflow    key = getpass("Roboflow Private API Key (hidden, or press Enter to skip): ").strip()    if key:        rf = Roboflow(api_key=key)        ds = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)\               .download("yolov8", location=str(ROOT / "dataset"), overwrite=True)        del key        val_dir = Path(ds.location) / "valid"        print(f"\nValidation set: {val_dir}")    else:        print("Skipped — no key entered.")else:    print("RUN_VALIDATION is False — skipping.")

In [ ]:
PALETTE = [(239, 68, 68), (34, 197, 94), (59, 130, 246), (234, 179, 8), (168, 85, 247)]def draw_gt(img_path, lbl_path, names):    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)    h, w = img.shape[:2]    n = 0    if lbl_path.exists():        for line in lbl_path.read_text().split("\n"):            p = line.split()            if len(p) < 5:                continue            c = int(p[0]); xc, yc, bw, bh = map(float, p[1:5])            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)            col = PALETTE[c % len(PALETTE)]            cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)            lab = names.get(c, str(c)) if isinstance(names, dict) else str(c)            cv2.putText(img, lab, (x1+3, max(12, y1-5)),                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2, cv2.LINE_AA)            n += 1    return img, nval_report = []if val_dir and val_dir.exists():    vimgs = sorted([p for p in (val_dir / "images").glob("*")                    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])    picks = random.sample(vimgs, min(N_VALIDATION, len(vimgs)))    print(f"Running on {len(picks)} of {len(vimgs)} validation images.\n")    for ip in picks:        lp = val_dir / "labels" / (ip.stem + ".txt")        gt_img, n_gt = draw_gt(ip, lp, model.names)        pr_img, res  = annotate(model, ip, CONF_THRESHOLD)        n_pr = len(res.boxes)        fig, ax = plt.subplots(1, 2, figsize=(14, 6))        ax[0].imshow(gt_img); ax[0].axis("off")        ax[0].set_title(f"Ground truth — {n_gt} object(s)", fontsize=11)        ax[1].imshow(pr_img); ax[1].axis("off")        ax[1].set_title(f"Prediction @ conf={CONF_THRESHOLD} — {n_pr} detection(s)", fontsize=11)        fig.suptitle(ip.name, fontsize=9)        plt.tight_layout()        plt.savefig(EVIDENCE / "validation" / f"val_{ip.stem}.png", dpi=110, bbox_inches="tight")        plt.show()        dets = [(model.names[int(c)], float(cf))                for c, cf in zip(res.boxes.cls, res.boxes.conf)]        val_report.append({"image": ip.name, "gt": n_gt, "pred": n_pr, "detections": dets})        delta = n_pr - n_gt        flag = "  <-- look at this one" if delta != 0 else ""        print(f"  {ip.name:<38} gt={n_gt:<3} pred={n_pr:<3} delta={delta:+d}{flag}")    print(f"\nSaved {len(picks)} comparisons -> results/evidence/validation/")    print("\nImages with delta != 0 are your error-analysis candidates.")else:    print("No validation set available — skipped.")

---## 6 · New-image inference — 5 unseen imagesImages the model has never seen and that were in no split. Performance here, not on validation, is what tells you whether this would survive contact with a real site.

In [ ]:
new_report = []if pool:    picks_new = pool[:N_NEW_IMAGES]    print(f"Running on {len(picks_new)} new image(s).\n")    for ip in picks_new:        im, res = annotate(model, ip, CONF_THRESHOLD)        out = EVIDENCE / "new_images" / f"pred_{ip.stem}.png"        plt.figure(figsize=(11, 8))        plt.imshow(im); plt.axis("off")        plt.title(f"{ip.name} — {len(res.boxes)} detection(s) @ conf={CONF_THRESHOLD}", fontsize=11)        plt.tight_layout(); plt.savefig(out, dpi=110, bbox_inches="tight"); plt.show()        dets = [(model.names[int(c)], round(float(cf), 3))                for c, cf in zip(res.boxes.cls, res.boxes.conf)]        new_report.append({"image": ip.name, "detections": dets})        print(f"  {ip.name}")        if dets:            for nm, cf in sorted(dets, key=lambda d: -d[1]):                print(f"      {nm:<24} {cf:.3f}")        else:            print("      nothing detected  <-- a false negative, if something is there")        print()    print(f"Saved -> results/evidence/new_images/")else:    print("No new images available. Add 5 to data/new_images/ and re-run.")

### 6.1 · Confidence threshold sweepThe governance checklist argues for **two thresholds**: a low one for progress logging (favouring recall, because a missed detection is silent) and a high one for site-condition alerts (favouring precision, because false alarms cause alert fatigue).This cell shows what each threshold actually does to the output, so the values you write into the governance checklist come from evidence rather than from the default.

In [ ]:
if pool:    ip = pool[0]    thresholds = [0.10, 0.25, 0.50, 0.75]    fig, axes = plt.subplots(1, len(thresholds), figsize=(6*len(thresholds), 6))    print(f"Sweep on: {ip.name}\n")    for ax, t in zip(axes, thresholds):        im, res = annotate(model, ip, t)        ax.imshow(im); ax.axis("off")        ax.set_title(f"conf = {t}  —  {len(res.boxes)} detection(s)", fontsize=11)        print(f"  conf={t:<5} -> {len(res.boxes)} detection(s)")    plt.tight_layout()    plt.savefig(EVIDENCE / "new_images" / "confidence_sweep.png", dpi=110, bbox_inches="tight")    plt.show()    print("\nLow threshold: more detections, more of them wrong.")    print("High threshold: fewer detections, more of them right, more misses.")    print("\n>>> Pick your two operating thresholds and record them in")    print(">>> docs/governance_checklist.md section 4.")else:    print("No images available for the sweep.")

---## 7 · Write the inference record and package the evidence

In [ ]:
import datetimerecord = {    "run_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",    "weights_url": WEIGHTS_URL,    "weights_sha256": sha,    "conf_threshold": CONF_THRESHOLD,    "iou_threshold": IOU_THRESHOLD,    "imgsz": IMGSZ,    "ultralytics": ultralytics.__version__,    "validation": val_report,    "new_images": new_report,}(RESULTS / "inference_record.json").write_text(json.dumps(record, indent=2))print("results/inference_record.json written.\n")print("EVIDENCE PACK")print("-" * 52)req = {"annotations": "3-5 (from notebook 01)",       "validation": f"{N_VALIDATION}",       "new_images": f"{N_NEW_IMAGES}",       "baseline": "comparison (bonus)"}for sub, need in req.items():    d = EVIDENCE / sub    n = len(list(d.glob("*"))) if d.exists() else 0    print(f"  {sub:<14} {n:>3} file(s)   required: {need}")print("-" * 52)zp = "/content/m4u3_evidence.zip"if os.path.exists(zp):    os.remove(zp)shutil.make_archive("/content/m4u3_evidence", "zip", root_dir=str(EVIDENCE))print(f"\n{zp}  ({os.path.getsize(zp)/1e6:.1f} MB)")try:    from google.colab import files    files.download(zp)except Exception as e:    print(f"(Auto-download unavailable: {e}) — use the Files panel on the left.")

---## Done**Next:**1. Unzip the evidence pack into `results/evidence/` in the repository and commit it.2. Open the validation comparisons and find your three false positives and three false negatives. Write them into `docs/error_analysis.md` **with filenames** — a hypothesis without an image behind it scores as guesswork.3. Read the two operating thresholds off the sweep in cell 6.1 and record them in `docs/governance_checklist.md` section 4.4. Check every link in the README resolves.**Reminder on what is being marked.** Reproducibility is 3 of the 10 points; model accuracy is 0 of them. A modest mAP in a repository that runs cleanly from a cold start beats a strong model that nobody else can re-run.